In [ ]:
# imports and settings

import os
import time
import pickle
import warnings
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from copy import deepcopy

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

%load_ext autoreload
%autoreload 2

# do not show warnings
warnings.filterwarnings("ignore")

print("Imports complete.")

import utils as ut

f0 = 1000
fs = 8000
duration = 60
snr = 0
fft_nperseg=16000
percent_overlap=0.25
window='hann'
remove_dc=20
crop_freq=2000
normalization_window_size=9

sim_signal = ut.simulate_raw_signal(f0=f0, fs=fs, duration=duration)
noisy_sim_signal = ut.add_noise_to_signal(signal=sim_signal, snr_db=snr, fs=fs, signal_bw=1)
wn = np.random.rand(len(sim_signal))
signals = [sim_signal, noisy_sim_signal, wn]

fig = make_subplots(rows=1, cols=3, column_widths=[0.3, 0.3, 0.4], column_titles=["Original", "Noisy", "White Noise"])

for ix, signal in enumerate(signals, start=1):
    F, T, Sxx, phase = ut.calc_spectrogram(signal, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

    # spectrogram
    fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showscale=False), row=1, col=ix)

    # phase distribution
    f_ix = np.argmin(np.abs(F - f0))
    f_phase = phase[f_ix, :]
    f_phase_wrapped = np.mod(f_phase, 2 * np.pi)   # np.angle returns [-pi, pi]; shift to [0, 2pi]
    


fig.update_layout(
    height=500, width=1500,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140),   # leave room for lifted column titles + shifted row titles
)
fig.update_xaxes(tickfont=dict(size=20), title_font=dict(size=20))
fig.update_yaxes(tickfont=dict(size=20), title_font=dict(size=20))

